## Example 1: this package is the one registrant of the two groups `opentelemetry-instrument` reads

In [ ]:
import importlib.metadata as md

def registrants(group):
    return sorted({(ep.dist.name.lower().replace("_", "-"), ep.name, ep.value) for ep in md.entry_points(group=group)})

# POSITIVE CONTROL for the reader: the SDK registers its console span exporter in a sibling group.
exporters = registrants("opentelemetry_traces_exporter")
assert ("opentelemetry-sdk", "console", "opentelemetry.sdk.trace.export:ConsoleSpanExporter") in exporters, exporters
distros, configurators = registrants("opentelemetry_distro"), registrants("opentelemetry_configurator")
print("opentelemetry_distro      :", distros)
print("opentelemetry_configurator:", configurators)
assert distros == [("opentelemetry-distro", "distro", "opentelemetry.distro:OpenTelemetryDistro")], distros
assert configurators == [("opentelemetry-distro", "configurator", "opentelemetry.distro:OpenTelemetryConfigurator")], configurators

## Example 2: `opentelemetry-instrument` exports a span to the console (with a negative control)

## Example 3: zero-code instrumentation of a plain `urllib` request

In [ ]:
import os, shutil, subprocess, sys

# Example 3: the child program never imports OpenTelemetry. `opentelemetry-instrument` loads the installed
# instrumentors (here python/opentelemetry_instrumentation_urllib), so its HTTP GET becomes a CLIENT span.
APP = """
import http.server, threading, urllib.request
srv = http.server.HTTPServer(("127.0.0.1", 0), http.server.SimpleHTTPRequestHandler)
threading.Thread(target=srv.serve_forever, daemon=True).start()
print(len(urllib.request.urlopen(f"http://127.0.0.1:{srv.server_port}/").read()), "bytes")
srv.shutdown()
"""
env = {**os.environ, "OTEL_TRACES_EXPORTER": "console", "OTEL_METRICS_EXPORTER": "none", "OTEL_LOGS_EXPORTER": "none"}
done = subprocess.run([shutil.which("opentelemetry-instrument"), sys.executable, "-c", APP], cwd="/tmp", env=env,
                      capture_output=True, text=True, timeout=180)
assert done.returncode == 0, done.stderr[-2000:]
print(done.stdout[:2000])
assert "bytes" in done.stdout, done.stdout
assert '"name": "GET"' in done.stdout and '"kind": "SpanKind.CLIENT"' in done.stdout, done.stdout[-2000:]


## Example 4: the distro's defaults (OTLP over gRPC), which never override a value already set

In [ ]:
import os
from opentelemetry.distro import OpenTelemetryDistro

# Example 4: `OpenTelemetryDistro.configure()` only SETS DEFAULTS in the environment: OTLP for traces, metrics and logs,
# over gRPC. A value the caller already chose (here the metrics exporter) is kept.
keys = ("OTEL_TRACES_EXPORTER", "OTEL_METRICS_EXPORTER", "OTEL_LOGS_EXPORTER", "OTEL_EXPORTER_OTLP_PROTOCOL")
saved = {key: os.environ.pop(key, None) for key in keys}
try:
    os.environ["OTEL_METRICS_EXPORTER"] = "console"
    OpenTelemetryDistro().configure()
    seen = {key: os.environ[key] for key in keys}
    print(seen)
    assert seen == {"OTEL_TRACES_EXPORTER": "otlp", "OTEL_METRICS_EXPORTER": "console",
                    "OTEL_LOGS_EXPORTER": "otlp", "OTEL_EXPORTER_OTLP_PROTOCOL": "grpc"}, seen
finally:
    for key, value in saved.items():
        os.environ.pop(key, None)
        if value is not None:
            os.environ[key] = value


## Example 5: the default `otlp` exporter resolves to the gRPC exporter the `otlp` extra ships

In [ ]:
import importlib.metadata as md

# Example 5: with the distro's defaults, the SDK looks up `otlp` in each exporter group. Those entry points come from
# python/opentelemetry_exporter_otlp_proto_grpc, which the distro's `otlp` extra brings in.
for group in ("opentelemetry_traces_exporter", "opentelemetry_metrics_exporter", "opentelemetry_logs_exporter"):
    by_name = {ep.name: ep for ep in md.entry_points(group=group)}
    assert "otlp" in by_name, (group, sorted(by_name))
    exporter_class = by_name["otlp"].load()
    print(f"{group}: otlp -> {exporter_class.__module__}.{exporter_class.__qualname__}")
    assert exporter_class.__module__.startswith("opentelemetry.exporter.otlp.proto.grpc."), exporter_class.__module__

from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

# Building the exporter opens no connection (gRPC connects lazily); nothing is sent before shutdown.
exporter = OTLPSpanExporter(endpoint="localhost:4317", insecure=True)
exporter.shutdown()
print("OTLPSpanExporter built and shut down")
